In [1]:
# import required libraries
from osgeo import gdal
from pathlib import Path
import geopandas as gpd
import subprocess

scotland = gpd.read_file('data/vectors/scotland/scotland.shp').to_crs(epsg=27700)

asc_folder = Path('data/rasters/scotland/ascs')
asc_folder.mkdir(parents=True, exist_ok=True)

warp_folder = Path('data/rasters/scotland/warped')
warp_folder.mkdir(parents=True, exist_ok=True)

dem_folder = Path('data/rasters/scotland/dems')
dem_folder.mkdir(parents=True, exist_ok=True)

footprint_folder = Path('data/vectors/scotland/tiles/footprints')
footprint_folder.mkdir(parents=True, exist_ok=True)

gpkg_folder = Path('data/vectors/scotland/tiles/gpkgs')
gpkg_folder.mkdir(parents=True, exist_ok=True)

In [2]:
for asc in asc_folder.rglob('*.asc'):
    if ((footprint_folder / asc.name)).with_suffix('.gpkg').exists():
        print(f'{asc.name} already processed. Skipping...')
        continue
    else:
        print(f'Finding footprint of {asc.name}...')
        gdal.Footprint(str((footprint_folder / asc.stem).with_suffix('.gpkg')), asc)
        print(f'Successfully found footprint of {asc.name}.')

for footprint in footprint_folder.iterdir():
    if (gpkg_folder / footprint.name).exists():
        print(f'{footprint.name} already processed. Skipping...')
        continue
    else:
        print(f'Processing {footprint.name}...')
        subprocess.run(['ogr2ogr', 
                        '-f', 
                        'GPKG', 
                        f'{gpkg_folder / footprint.name}', 
                        f'{footprint}'], check=True)
        print(f'Successfully processed {footprint.name}.')

for gpkg in gpkg_folder.iterdir():
    shape = gpd.read_file(gpkg).to_crs(epsg=27700)
    if shape.intersects(scotland.union_all())[0] != True:
        (asc_folder / gpkg.stem).with_suffix('.asc').unlink()
        (asc_folder / gpkg.stem).with_suffix('.prj').unlink()


NN17NE.asc already processed. Skipping...
NG76NE.asc already processed. Skipping...
NT13NW.asc already processed. Skipping...
NX66NE.asc already processed. Skipping...
NO40NE.asc already processed. Skipping...
NG43SW.asc already processed. Skipping...
NT70NW.asc already processed. Skipping...
HU67SW.asc already processed. Skipping...
NT56SE.asc already processed. Skipping...
NS35SW.asc already processed. Skipping...
NT00NW.asc already processed. Skipping...
NA81SE.asc already processed. Skipping...
NO77NE.asc already processed. Skipping...
NG26NW.asc already processed. Skipping...
NM65NE.asc already processed. Skipping...
NO12NE.asc already processed. Skipping...
NR99SW.asc already processed. Skipping...
NN35NE.asc already processed. Skipping...
NS29SE.asc already processed. Skipping...
HU43NE.asc already processed. Skipping...
NX06SW.asc already processed. Skipping...
NH95SW.asc already processed. Skipping...
NB91SE.asc already processed. Skipping...
NH06SW.asc already processed. Skip

In [3]:
for asc in asc_folder.rglob('*.asc'):
    if (warp_folder / asc.name).exists():
        print(f'{asc.name} already processed. Skipping...')
        continue
    else:
        print(f'Reprojecting {asc.name}...')
        gdal.Warp(str(warp_folder / asc.name), asc, dstSRS='EPSG:27700')
        print(f'{asc.name} reprojected successfully.')

        # TODO add cutline to trim boundary to Scotland.

Reprojecting NN17NE.asc...


/home/jamiemac/miniconda3/envs/geospatial/lib/python3.14/site-packages/osgeo/gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


NN17NE.asc reprojected successfully.
Reprojecting NG76NE.asc...
NG76NE.asc reprojected successfully.
Reprojecting NT13NW.asc...
NT13NW.asc reprojected successfully.
Reprojecting NX66NE.asc...
NX66NE.asc reprojected successfully.
Reprojecting NO40NE.asc...
NO40NE.asc reprojected successfully.
Reprojecting NG43SW.asc...
NG43SW.asc reprojected successfully.
Reprojecting NT70NW.asc...
NT70NW.asc reprojected successfully.
Reprojecting NT56SE.asc...
NT56SE.asc reprojected successfully.
Reprojecting NS35SW.asc...
NS35SW.asc reprojected successfully.
Reprojecting NT00NW.asc...
NT00NW.asc reprojected successfully.
Reprojecting NO77NE.asc...
NO77NE.asc reprojected successfully.
Reprojecting NG26NW.asc...
NG26NW.asc reprojected successfully.
Reprojecting NM65NE.asc...
NM65NE.asc reprojected successfully.
Reprojecting NO12NE.asc...
NO12NE.asc reprojected successfully.
Reprojecting NR99SW.asc...
NR99SW.asc reprojected successfully.
Reprojecting NN35NE.asc...
NN35NE.asc reprojected successfully.
Rep

In [4]:
for file in warp_folder.rglob('*.asc'):
    if Path(dem_folder / file.stem).with_suffix('.tif').is_file():
        print(f'{file.name} already processed. Skipping...')
        continue
    else:
        print(f'Converting {file.name}...')
        gdal.Translate((dem_folder / file.stem).with_suffix('.tif'), file, format='GTiff')
        print(f'{file.name} converted successfully.')


Converting NN17NE.asc...
NN17NE.asc converted successfully.
Converting NG76NE.asc...
NG76NE.asc converted successfully.
Converting NT13NW.asc...
NT13NW.asc converted successfully.
Converting NX66NE.asc...
NX66NE.asc converted successfully.
Converting NO40NE.asc...
NO40NE.asc converted successfully.
Converting NG43SW.asc...
NG43SW.asc converted successfully.
Converting NT70NW.asc...
NT70NW.asc converted successfully.
Converting NT56SE.asc...
NT56SE.asc converted successfully.
Converting NS35SW.asc...
NS35SW.asc converted successfully.
Converting NT00NW.asc...
NT00NW.asc converted successfully.
Converting NO77NE.asc...
NO77NE.asc converted successfully.
Converting NG26NW.asc...
NG26NW.asc converted successfully.
Converting NM65NE.asc...
NM65NE.asc converted successfully.
Converting NO12NE.asc...
NO12NE.asc converted successfully.
Converting NR99SW.asc...
NR99SW.asc converted successfully.
Converting NN35NE.asc...
NN35NE.asc converted successfully.
Converting NS29SE.asc...
NS29SE.asc conv

In [ ]:
scotland = gpd.read_file('data/vectors/scotland/scotland.shp')
test = gpd.read_file('data/vectors/scotland/tiles/trial.gpkg').to_crs(epsg=27700)

ax = scotland.plot()
test.plot(ax=ax, fc='red')

In [5]:
print(len([asc for asc in asc_folder.rglob('*.asc')]))

3920
